## Producer-Consumer Pattern
This pattern involves two types of threads: producers generating data and consumers processing that data. A blocking queue acts as a buffer between the two.

asyncio.Queue() in Python provides an asynchronous, thread-safe, and first-in, first-out (FIFO) queue 
specifically designed for use within asyncio applications. It enables synchronized communication and data exchange 
between different coroutines. 

Key characteristics and usage: 

1.Asynchronous Operations: The put() and get() methods of asyncio.Queue are awaitable. This means a coroutine attempting to get() an item from an empty queue will pause its execution until an item becomes available, without blocking the entire event loop. Similarly, a put() operation on a full queue (if a maxsize was specified) will also await until space becomes available. 

2. Synchronization Primitive: It acts as a crucial synchronization primitive, allowing producer coroutines to add items to the queue and consumer coroutines to retrieve them in a coordinated manner. 

3. maxsize Parameter: When creating an asyncio.Queue, an optional maxsize argument can be provided. This limits the number of items the queue can hold, preventing unbounded memory consumption. If maxsize is 0 (the default), the queue size is unlimited. 

4. Methods:
 
	1. await put(item): Adds an item to the queue. If the queue is full (and maxsize is set), this operation will await until space is available. 

	2. await get(): Removes and returns an item from the queue. If the queue is empty, this operation will await until an item is available. 
	
	3. qsize(): Returns the current number of items in the queue. 
	
	4• empty(): Returns True if the queue is empty, False otherwise. 
	
	5• full(): Returns True if the queue is full (based on maxsize), False otherwise. 

In [ ]:
import asyncio

async def producer(queue):
    for i in range(5):
        await asyncio.sleep(0.1) # Simulate some work
        await queue.put(f"Item {i}")
        print(f"Produced: Item {i}")

async def consumer(queue, consumer_id):
    while True:
        item = await queue.get()
        print(f"Consumer {consumer_id} consumed: {item}")
        queue.task_done() # Indicate that a retrieved task has been processed

async def main():
    queue = asyncio.Queue(maxsize=3) # Create a queue with a max size
    producers = [asyncio.create_task(producer(queue))]
    consumers = [asyncio.create_task(consumer(queue, i)) for i in range(2)]

    await asyncio.gather(*producers) # Wait for producers to finish
    await queue.join() # Wait until all items in the queue have been processed
    for c in consumers:
        c.cancel() # Cancel consumer tasks

if __name__ == "__main__":
    asyncio.run(main())

In [ ]:
import threading
import queue
import time
import random

# The shared queue object handles all necessary locking internally
shared_queue = queue.Queue(maxsize=10)
NUM_ITEMS = 10

def producer():
    """Produces items and adds them to the queue."""
    for i in range(NUM_ITEMS):
        item = random.randint(1, 100)
        shared_queue.put(item) # put() adds an item, blocks if the queue is full
        print(f"Producer produced item {item} (Queue size: {shared_queue.qsize()})")
        time.sleep(random.random() * 0.5) # Simulate production time
    
    # Add a sentinel value to signal completion
    shared_queue.put(None)
    print("Producer finished all items and sent stop signal.")

def consumer():
    """Consumes items from the queue."""
    while True:
        item = shared_queue.get() # get() removes and returns an item, blocks if the queue is empty
        if item is None:
            # Check for the sentinel value to exit the loop
            print("Consumer received stop signal, exiting.")
            break
        print(f"Consumer got => {item}")
        time.sleep(random.random() * 0.5) # Simulate consumption time
        shared_queue.task_done() # Mark the task as done

# Main part of the program
if __name__ == "__main__":
    producer_thread = threading.Thread(target=producer)
    consumer_thread = threading.Thread(target=consumer)

    producer_thread.start()
    consumer_thread.start()

    # Wait for the producer to finish
    producer_thread.join()
    
    # Wait for all items in the queue to be processed
    # shared_queue.join() blocks until all items have been consumed and task_done() called
    # Note: Using shared_queue.join() in this specific example with a single sentinel 
    # might not be necessary, as the consumer loop breaks upon receiving the sentinel.
    # consumer_thread.join() is sufficient to wait for the consumer to exit its loop.
    consumer_thread.join() 

    print("Main thread exiting.")
